#Data Engineering + GenAI
Phase 2 begins
---
Phase 1 built the data pipeline, phase 2 adds AI on top

today's Pipeline


```
Messy Invoice text
      |
[PROMPT ENGINEERING]  design precise instructions
      |
 [GROQ API]    Call LLm(Llama 3.1)
      |
  [JSON Parsing]  Extract structured data
      |
 [PANDAS DATAFROME] Clean, analysable table
       |

  [ANALYSIS]     business analysis     
```



In [ ]:
#CELL 1
!pip install groq --quiet

import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

print('Libraries imported successfully')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.3 MB/s eta 0:00:00
Libraries imported successfully


In [ ]:
#CELL 2
from groq import Groq
#Import the groq client class from the groq library

API_KEY='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'

#REPACE with your groq API Key from console.groq.com
#Security : never share this key or commit it to github

client=Groq(api_key=API_KEY)
#crate a configured Groq client
#This client handles authentication and HTTP details automatically
#All API calls go through this client objects
#Initialize the Groq client with your API key

MODEL="llama-3.1-8b-instant"
#The LLM model to use
#'llama-3.1-8b-instant'  =  meta's Llama 3.1, 8 billion parameters, fast
#Other options: 'mistral-8x7b-32768' , 'llama-3.1-70b-versatile'

print(f'Groq client configured with model: {MODEL}')
print('Make sure API_KEY is replaced with your actual key')
#

Groq client configured with model: llama-3.1-8b-instant
Make sure API_KEY is replaced with your actual key


In [ ]:
#CELL 3

def ask_llm(user_message, system_message='You are a helpful assistant.',temperature=0.7, max_tokens=500):
    """
    Send a message to the LLM and return the response text.

    Parameters:
    ----------------------
    user_message :
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            #system messages = instructions to the modek about how to behave
            #Like telling a new employee: 'You work at teh data company'
            {"role": "user", "content": user_message},
        ],
        temperature=temperature,
        #temperature controls randomness
        #0.0 = always picks the most likely next token (consisten)
        max_tokens=max_tokens
        #maximum number od tokens in the response
        #500 tokens = 375 words
    )
    return response.choices[0].message.content
    #response.choices = list of possible completions (we requested 1)
    #[0]              = first completion
    #.message.content = the actual text the LLM generated

#Test the connection
test_response=ask_llm(
    "What is ETL in data engineering? Answer in exactly 2 sentences."
)
test_response2 = ask_llm(
    "Is GenAI abd Data Engineering a good career in 2026?"
)
print('=== LLM Response')
print(test_response)
print(test_response2)

=== LLM Response
ETL (Extract, Transform, Load) is a process in data engineering that involves extracting data from various sources, transforming it into a standardized format, and loading it into a target system, such as a data warehouse or a database, for analysis and storage. The ETL process is commonly used to integrate data from multiple sources, resolve data inconsistencies, and ensure data quality, making it a crucial step in data engineering pipelines.
As of my cut-off knowledge in 2023, GenAI (General Artificial Intelligence) and Data Engineering are rapidly growing fields with high demand. Here's a breakdown of their prospects in 2026:

**GenAI:**

1. **Growing demand:** With the increasing adoption of AI in various industries, the need for skilled professionals in GenAI is expected to rise. This field involves developing and applying AI models to solve complex problems.
2. **High salary potential:** GenAI professionals are among the highest-paid in the industry, with salarie

In [ ]:
#CELL 4
#ask about concepts from phase 1- showing LLm knows our domain
response_etl=ask_llm(
    "In 3 bullet points, explain how the medallion Architecture "
    "(Bronze, Silver, Gold layers) realtes to ETL pipeline.",
    system_message="You are a senior data engineering instruction. "
                   "Be concise and practical."
)

print('Medallion + ETL connection:')
print(response_etl)
print()
print('--- Token Explanation ---')
print('Each word is roughly 1-2 tokens.')
print('The model above used approximately',len(response_etl.split())*1.3,'tokens.')
print('Llama-3.1-8b content window: 8192 tokens (~6000 words per conversation)')

Medallion + ETL connection:
Here are 3 bullet points explaining how the Medallion Architecture (Bronze, Silver, Gold layers) relates to an ETL pipeline:

• **Bronze Layer (Raw Data)**: This layer represents the source system's raw data, which is the starting point of the ETL pipeline. Data is stored in its original format and is typically unprocessed and untransformed. The bronze layer acts as a landing zone for data ingestion.

• **Silver Layer (Raw Processed Data)**: After initial processing, the data is moved to the silver layer, where it's transformed to a standard format and cleansed of errors. This layer represents a semi-processed state, where data is more structured but not yet finalized. The silver layer is where data is typically denormalized and prepared for further processing.

• **Gold Layer (Transformed Data)**: The gold layer represents the final, transformed data that's ready for consumption by users or applications. This layer is the output of the ETL pipeline, where d

# ACTIVITY 2:

#EXPERMENT 1 : ZERO SHOT PROMPTING

In [ ]:
#CELL 5
zero_shot_response=ask_llm(
    "Extract the city name from this address: "
    "456 Brigade Road, Bangalore 560025, Karnataka, India"
)
print('Zero-Shot Result:')
print(zero_shot_response)
print()

#zero-shot works well here because 'extract city' is clear
#Let's try a more ambiguous task
ambiguous_response = ask_llm("Clean this data: ramesh kumar,45000,mumbai")
print(ambiguous_response)
print()
print('Problem: output format is unpredictable and not machine-parsable!')

Zero-Shot Result:
The city name is Bangalore.

To clean the data, I will break it down into individual fields and provide a cleaned version of each field.

Original Data: 
ramesh kumar, 45000, mumbai

Cleaned Data:

- Name: Ramesh Kumar (I changed the format to title case, where the first letter of each word is capitalized)
- Salary: $45,000 (I added a dollar sign for clarity and to indicate that it's a monetary value)
- Location: Mumbai (no changes needed, as it's already in a clear and concise format)

Cleaned Data: 
Ramesh Kumar, $45,000, Mumbai

Problem: output format is unpredictable and not machine-parsable!


#EXPERIMENT : 2

In [ ]:
#CELL 6
few_shot_prompt="""
Convert employee text to JSON.Here are examples:

Input: RAMESH KUMAR, 45000, mumbai
Output: {"name": 'RAMESH KUMAR', "salary": 45000, "city": 'mumbai'}

Input: priya nair, 52000, Delhi
Output: {"name": 'priya nair', "salary": 52000, "city": 'Delhi'}

Now convert this
Input: ANANYA DAS, 38000, kolkata
Output:"""
#The examples teach the model
few_shot_response=ask_llm(few_shot_prompt, temperature=0.0)
#temperature=0.0 for maximum consistency
print('Few-Shot Result:')
print(few_shot_response)
print()


#try parsing it
try:
  parsed=json.loads(few_shot_response.strip())
  print('Successfully parsed as JSON!')
  print(f"Name: {parsed['name']}")
  print(f"Salary: {parsed['salary']}")
  print(f"City: {parsed['city']}")
except json.JSONDecodeError:
  print('Parsing failed - model added extra text')
  print('Solution: add explicit instruction in the system prompt')
  print('Failed to parse as JSON!')

Few-Shot Result:
Here's a Python function that can convert the employee text to JSON:

```python
import json

def employee_to_json(name, salary, city):
    return {
        "name": name,
        "salary": salary,
        "city": city
    }

# Test the function
name = "ANANYA DAS"
salary = 38000
city = "kolkata"

employee_json = employee_to_json(name, salary, city)
print(json.dumps(employee_json, indent=4))
```

When you run this code, it will output:

```json
{
    "name": "ANANYA DAS",
    "salary": 38000,
    "city": "kolkata"
}
```

However, if you want to directly convert the input string to JSON without manually splitting it into name, salary, and city, you can use the following function:

```python
import json

def employee_text_to_json(text):
    parts = text.split(', ')
    return {
        "name": parts[0],
        "salary": int(parts[1]),
        "city": parts[2]
    }

# Test the function
text = "ANANYA DAS, 38000, kolkata"
employee_json = employee_text_to_json(text)
print(j

#EXPERIMENT : 3

In [ ]:
#CELL 7

same_question = """Review this python code and identify any issues:
df['revenue']=df['price']*df['quantity']
result=df.groupby('dept').sum()"""

#WITHOUT ROLE

generic_response=ask_llm(same_question,temperature=0.2)
print('Without Role Prompting')
print(generic_response[:300],'...')
print()

#WITH ROLE
role_response=ask_llm(
    same_question,
    system_message="You are  a senior data engineer with 10 years of production "
                   "experience. Review code crictically for production readiness,"
                   "data type issues, and potential failures at scale.",
    temperature=0.2
)
print('With Role Prompting (senior data engineer):')
print(role_response[:400],'...')
print()
print('Notice: role prompting produces more technical, actionable feedback')

Without Role Prompting
The provided Python code appears to be a simple data manipulation task using the pandas library. However, there are a few potential issues that can be identified:

1. **Missing Error Handling**: The code does not include any error handling. If the 'price' or 'quantity' columns do not exist in the Da ...

With Role Prompting (senior data engineer):
**Code Review**

The provided Python code appears to be a simple data manipulation task using the pandas library. However, there are a few potential issues that could impact production readiness:

```python
# Assuming df is a pandas DataFrame
df['revenue'] = df['price'] * df['quantity']
result = df.groupby('dept').sum()
```

**Potential Issues:**

1. **Data Type Issues:**
   - The code assumes tha ...

Notice: role prompting produces more technical, actionable feedback


#EXPERIMENT : 4 -- TEMPERATURE EFFECT

In [ ]:
#CELL 8

prompt='Give me one creative name for a data analytics startup'

print('=== Temperature Experiment ===')
for temp in [0.0, 0.5, 1.0]:
    response=ask_llm(prompt, temperature=temp)
    print(f'Temperature={temp}: {response.strip()}')
    time.sleep(1)        #Small pause to respect rate limits

print()
print('Observation')
print(' temperature=0.0 -> same or very similar answer every run (deterministic)')
print(' temperature=0.5 -> some variation')
print(' temperature=1.0 -> more creative/varied, sometimes surprising')
print('Rule for data engineering tasks: use temperature=0.0 or 0.1')
print('You need CONSISTENT, PARSABLE output - not creative variation')

=== Temperature Experiment ===
Temperature=0.0: Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys a sense of innovation and forward-thinking, which is perfect for a data analytics startup.
Temperature=0.5: Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connections and networks, implying the ability to connect and analyze data from various sources. "Insights" conveys the idea of gaining valuable knowledge and understanding from data analysis, which is the core mission of a data analytics startup.
Temperature=1.0: One creative name for a data analytics startup could be "Nexus Insights". 

"Nexus" suggests a connection or a link between different pieces of information, which is what data analysis often strives for. "Insights" implies the provision of valuable information and understa

In [ ]:
#CELL 9 Weak vs Strong Prompt Comparison

invoice_text = "Invoice #2024-001 from TECHWORLD SOLUTIONS dates 15th January 2024. Amount: Rs.45,000 for Laptop"

#WEAK PROMPT  -  vague, no format specification
weak_response=ask_llm(
    f"Clean this invoice data: {invoice_text}",
    temperature=0.3
)
print('Weak PROMPT OUTPUT:')
print(weak_response)
print()

#Test if parsable
#Try to convert the LLM response into valid JSON
#If the response is properly formatted JSON, json.loads() will succeed
try:
  json.loads(weak_response)
  #Executed only if JSON parsing succeeds
  #This means the response can potentially be loaded into a DataFrame
  print('PARSABLE: Yes')

except:
  print('PARSABLE: No - cannot load into a DataFrame')

print('\n'+ '='*50 +'\n')


#STRONG PROMPT - role + schema + fromat contraint
strong_system = """You are a data extraction specialist for an acconding pipeline.
Extract invoice data and return ONLY a valid JSON object.
Do NOT include any explanation, preamble, or markdown formatting.
Return ONLY the JSON, nothing ELSE

JSON scheama (use null for missing values):
{"invoice_id":string, "vendor_name":string (Title Case),
"amount":number (no currency symbols),
"currency":string (default INR),
"invoice_date":string (YYYY-MM-DD),
"category":string (Electronics/Services/Accessories/Other)}"""

strong_response=ask_llm(
    f"Extract from: {invoice_text}",
    system_message=strong_system,
    temperature=0.0      #Always 0 for structured data extraction
)
print('Strong PROMPT OUTPUT:')
print(strong_response)
print()

try:
  parsed=json.loads(strong_response.strip())
  print('PARSABLE :Yes')
  print(f'Vendor :{parsed.get("vendor_name")}')
  print(f'Amount: {parsed.get("amount")}')
  print(f'Date: {parsed.get("invoice_date")}')
except json.JSONDecodeError:
  #Fallback: extract JSON with regex
  match=re.search(r'\{.*?\}', strong_response,re.DOTALL)
  if match:
    parsed=json.loads(match.group())
    print('PARSABLE: Yes (extracted with regex fall back)')
  else:
    print('Parsable : no - retry with stricket prompt')

Weak PROMPT OUTPUT:
Here's the cleaned invoice data:

**Invoice Details:**

- **Invoice Number:** 2024-001
- **Date:** January 15, 2024
- **Supplier:** Techworld Solutions
- **Item Purchased:** Laptop
- **Amount:** Rs. 45,000

Let me know if you'd like me to format it differently or if you have any other requests.

PARSABLE: No - cannot load into a DataFrame


Strong PROMPT OUTPUT:
{"invoice_id": "2024-001", "vendor_name": "Techworld Solutions", "amount": 45000, "currency": "INR", "invoice_date": "2024-01-15", "category": "Electronics"}

PARSABLE :Yes
Vendor :Techworld Solutions
Amount: 45000
Date: 2024-01-15


#MINI PROJECT :Smart Data Cleaner
**Goal**: Convert 5 messy invoice strings into a clean, structured Pandas DataFrame using LLM.

This is a complete GenAI-powered ETL pipeline

```Messy Text -> LLM -> JSON ->DataFrame -> Analysis```

**Q1:** What is the Difference between ML (Day 5) and generative AI(Day6)?

**Q2:**what does ```temperature=0.0``` do in a LLM API call and when would you use it?

**Q3:**Write a few -shot prompt that extracts name annd salary from text in JSON format.

**Q4:**What is LLM hallucination and how can prompt engineering reduce it?

**Q5:**Your LLM return ```json\n{"name":"Ramesh"}\n``` and ```json.loads()``` crashes. Write the fix.

**Q6:**How does today's

#### Q1. Difference between ML and Generative AI
**ML** focuses on learning patterns from data to make predictions or decisions.


**Generative AI** is a type of ML that creates new, original content (like text or images) based on learned patterns.

*   **Machine Learning (ML)** typically focuses on tasks like classification, regression, and prediction based on existing data. It learns patterns from data to make decisions or forecasts.
*   **Generative AI (GenAI)**, a subset of ML, focuses on creating new, original content (like text, images, or code) that is similar to the data it was trained on but not identical. It generates rather than just analyzes or predicts.

**Q2: What does `temperature=0.0` do in an LLM API call and when would you use it?**

*   `temperature=0.0` makes the LLM's output **deterministic**. It means the model will consistently pick the most probable next token, resulting in the same or very similar answer for the same prompt every time.
*   You would use `temperature=0.0` (or a very low value like `0.1`) when you need **consistent, predictable, and machine-parsable output**, such as for structured data extraction, code generation, or when reliability and accuracy are paramount over creativity.

`temperature=0.0` makes the LLM's output highly deterministic and consistent (least creative). Use it for tasks requiring precise, repeatable results, such as data extraction and parsing.

#### Q3. Few-shot prompt for name and salary extraction in JSON

In [ ]:
few_shot_prompt_short = '''
Convert employee text to JSON. Examples:
Input: John Doe, 60000
Output: {"name": "John Doe", "salary": 60000}

Input: Jane Smith, 75000
Output: {"name": "Jane Smith", "salary": 75000}

Input: ALICE JOHNSON, 82000
Output:
'''
print('Short Few-Shot Prompt:')
print(few_shot_prompt_short)
# Expected output for 'ALICE JOHNSON, 82000' would be: {"name": "Alice Johnson", "salary": 82000}

Short Few-Shot Prompt:

Convert employee text to JSON. Examples:
Input: John Doe, 60000
Output: {"name": "John Doe", "salary": 60000}

Input: Jane Smith, 75000
Output: {"name": "Jane Smith", "salary": 75000}

Input: ALICE JOHNSON, 82000
Output:



#### Q4. LLM hallucination and how prompt engineering reduces it
**LLM hallucination** is when the model generates false or nonsensical information. **Prompt engineering** reduces it by providing clear instructions, context, examples, and strict output constraints (e.g., "Answer only from provided text", "Return ONLY JSON").


*   **LLM Hallucination** refers to when a large language model generates information that is plausible-sounding but factually incorrect, nonsensical, or fabricated, without evidence in its training data or the provided context.
*   **Prompt engineering can reduce it by:**
    *   **Providing clear, specific instructions:** Guiding the model to stay within defined boundaries.
    *   **Using system prompts:** Setting a precise role and behavior for the model (e.g., "You are a data extraction specialist, return only facts").
    *   **Few-shot prompting:** Giving examples of desired input-output pairs to show the model the expected format and content.
    *   **Grounding the model:** Including relevant information directly in the prompt so the model relies on the provided context rather than generating its own.
    *   **Setting `temperature` to a low value (e.g., 0.0 or 0.1):** This makes the model's responses less creative and more focused on the most probable and factual tokens.

#### Q5. Fix for `json.loads()` crashing on ```json\n{"name":"Ramesh"}\n

In [ ]:
import json
import re

llm_output_bad = '```json\n{"name":"Ramesh"}\n```'

# Use regex to extract the pure JSON string
match = re.search(r'```json\n(.*)\n```', llm_output_bad, re.DOTALL)
if match:
    json_string_fixed = match.group(1).strip()
    try:
        parsed_data = json.loads(json_string_fixed)
        print(f"Fixed: Successfully parsed as: {parsed_data}")
    except json.JSONDecodeError as e:
        print(f"Error even after regex: {e}")
else:
    print("No JSON block found.")

Fixed: Successfully parsed as: {'name': 'Ramesh'}


**Q6: How does today's pipeline relate to GenAI (Phase 2)?**

Today's pipeline, as described in the notebook, is about integrating **Generative AI (GenAI)**, specifically using LLMs like Llama 3.1, into a data engineering workflow. It demonstrates how GenAI is used for:

*   **Prompt Engineering:** Designing instructions for the LLM.
*   **LLM Invocation:** Calling the Groq API to get responses from the LLM.
*   **Structured Data Extraction:** Using LLMs to convert messy, unstructured text (like invoice text) into structured, machine-parsable formats (JSON).
*   **Data Preparation for Analysis:** Taking the LLM's structured output and converting it into a Pandas DataFrame for cleaning and business analysis.

In essence, Phase 2 **adds GenAI capabilities** on top of the data pipeline established in Phase 1 (which built the data pipeline), turning unstructured data into structured, analyzable information using advanced language models.

#### Q6. Smart Cleaner vs. Manual ETL Cleaning
Today's **Smart Cleaner** (LLM-based) automatically extracts and structures data from *unstructured text* using natural language understanding, adapting flexibly to variations. **Manual ETL** relies on predefined rules for *structured/semi-structured data* and requires more effort to adapt to new formats.

In [ ]:
# Make sure to run all preceding cells, especially the one defining `ask_llm`.
few_shot_prompt = """
Convert employee text to JSON. Here are examples:

Input: RAMESH KUMAR, 45000, mumbai
Output: {"name": "Ramesh Kumar", "salary": 45000, "city": "Mumbai"}

Input: priya nair, 52000, Delhi
Output: {"name": "Priya Nair", "salary": 52000, "city": "Delhi"}

Now convert this:
Input: ANANYA DAS, 38000, kolkata
Output:
"""
few_shot_response = ask_llm(few_shot_prompt, temperature=0.0)
print('Few-Shot Result:')
print(few_shot_response)
print()

try:
  # The type hint 'parsed: json.loads' is incorrect. It should be 'parsed = json.loads'
  parsed = json.loads(few_shot_response.strip())
  print('Sucessfully parsed as JSON!')
  print(f'Name: {parsed["name"]}, Salary: {parsed["salary"]}, City: {parsed["city"]}')

except json.JSONDecodeError:
  print('Parsing failed - model added extra text or invalid JSON.')
  print("Solution: add explicit instructions in the system prompt to return ONLY JSON, and check your prompt examples for correct JSON syntax.")


Few-Shot Result:
Here's a Python function that can convert the employee text to JSON:

```python
import json

def convert_to_json(employee_text):
    # Split the input string into individual values
    values = employee_text.split(', ')
    
    # Create a dictionary with the given keys
    employee_data = {
        "name": values[0].strip().title(),
        "salary": int(values[1]),
        "city": values[2].strip().title()
    }
    
    # Convert the dictionary to JSON
    json_data = json.dumps(employee_data)
    
    return json_data

# Test the function
employee_text = "ANANYA DAS, 38000, kolkata"
print(convert_to_json(employee_text))
```

When you run this function with the input "ANANYA DAS, 38000, kolkata", it will output:

```json
{"name": "Ananya Das", "salary": 38000, "city": "Kolkata"}
```

This function works by splitting the input string into individual values using the `split(', ')` method. It then creates a dictionary with the given keys and assigns the corresponding v